<img src="banner.png" width="100%" style="max-height:300px; object-fit:cover;"/>

# Curso de Agentes de IA con LangGraph

Este notebook sirve como una guía introductoria para el diseño e implementación de **Agentes de Inteligencia Artificial** utilizando el framework **LangChain** y la integración de la familia **Gemini** a través de **Google Cloud Vertex AI**.

A lo largo de este curso, aprenderemos a:

- Conectar modelos de lenguaje con herramientas externas.
- Crear flujos de trabajo cíclicos basados en grafos.
- Dotar a los sistemas de capacidades de razonamiento.
- Diseñar agentes con toma de decisiones autónoma.

> El objetivo es construir una base sólida para desarrollar agentes inteligentes capaces de interactuar con herramientas, procesar información y ejecutar tareas de forma dinámica.


## Paso 1: Instalación de Dependencias

Para comenzar, instalaremos todas las librerías necesarias para el curso en su versión más reciente. Esto incluye:

- **langchain**, **langchain-community**, **langchain-google-genai**: módulos principales y de integración con Google Generative AI (Gemini).
- **langgraph**: motor para crear agentes basados en grafos con estados y ciclos.
- **python-dotenv**: carga de variables de entorno desde un archivo `.env`.
- **arxiv**: herramienta de búsqueda y consulta académica para ejercicios del curso.

> **Nota:** usamos el parámetro `-q` (*quiet*) para realizar una instalación limpia y libre de barras de progreso ruidosas.


In [1]:
# Instalar dependencias del curso de forma silenciosa
!pip install -q -U langchain langchain-community langgraph langchain-google-genai langchain-tavily google-auth python-dotenv arxiv tavily-python

## Paso 2: Autenticación con Google Cloud (ADC)

Para conectarnos a **Vertex AI** sin usar una API key, utilizamos **Application Default Credentials (ADC)**. Este mecanismo detecta automáticamente las credenciales del entorno local.

Ejecuta los siguientes comandos **una sola vez** desde la terminal integrada de DataSpell (`Alt + F12`):

```bash
# Iniciar sesión con tu cuenta de Google
gcloud auth login

# Generar el archivo de credenciales ADC en tu entorno local
gcloud auth application-default login

# Establecer tu proyecto de Google Cloud (debe tener Vertex AI habilitado)
gcloud config set project TU_PROJECT_ID
```

Las credenciales se almacenan en `%APPDATA%\gcloud\application_default_credentials.json` y son detectadas automáticamente por el SDK.

> **Nota:** el `project_id` se carga desde un archivo `.env` para no hardcodearlo en el código.


## Paso 3: Configuración del Entorno e Importación de Librerías

En este paso, configuraremos el entorno de Python e importaremos las librerías esenciales para trabajar con **LangChain** y **LangGraph**.

Esto incluye:

- **warnings**: módulo utilizado para controlar y filtrar advertencias durante la ejecución del notebook.
- **dotenv**: carga del `GOOGLE_CLOUD_PROJECT` y `TAVILY_API_KEY` desde el archivo `.env`.
- **google.auth**: obtención de credenciales ADC para autenticación con Google Cloud sin API key.
- **langchain-google-genai**: integración de LangChain con los modelos Gemini de Google.
- **LangGraph**: extensión que permite crear flujos de trabajo basados en grafos, estados y ciclos.

> **Nota:** configuraremos un filtro de advertencias para mantener la salida del notebook limpia, evitando mensajes de depreciación o avisos menores que no afecten el desarrollo del curso.


In [8]:
import os
import warnings
from dotenv import load_dotenv

load_dotenv()

warnings.filterwarnings("ignore")
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["PYTHONWARNINGS"] = "ignore"

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
import langgraph

print("✅ Entorno configurado e importaciones listas.")

✅ Entorno configurado e importaciones listas.


## Paso 4: Inicialización del Modelo, Herramientas y Prueba de Conexión

En este paso, inicializaremos el modelo **Gemini 2.5 Flash**, definiremos las herramientas del agente y realizaremos una prueba de conexión.

Esto incluye:

- **Inicialización del modelo**: creación de una instancia de `ChatGoogleGenerativeAI` autenticada mediante ADC con Vertex AI.
- **busca_web**: herramienta que realiza búsquedas en la web usando Tavily Search.
- **create_react_agent**: creación del agente con `system_prompt` y herramientas vinculadas.
- **Prueba de conexión**: verificación de que las credenciales ADC y la configuración del entorno están funcionando.

> **Nota:** esta prueba permite confirmar que el entorno está listo para comenzar a trabajar con modelos de lenguaje dentro de flujos creados con LangChain y LangGraph.


In [9]:
import google.auth
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# Obtener credenciales ADC y project_id
credentials, project_id = google.auth.default()

# Inicializar el modelo Gemini 2.5 Flash con credenciales ADC
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    credentials=credentials
)

@tool
def busca_web(query: str) -> list:
    """Realiza una búsqueda en la web sobre un tema específico"""
    tavily_search = TavilySearch(max_results=5)
    return tavily_search.invoke(query)

tools = [busca_web]

system_prompt = """
Actúa como un asistente útil y especializado en investigación.
Utiliza las herramientas proporcionadas para responder a las preguntas del usuario.

Herramientas disponibles:
- busca_web: realiza búsquedas en la web y devuelve enlaces y resúmenes.

Siempre que el usuario pregunte sobre un tema específico:
1. Usa la herramienta busca_web.
2. Analiza los resultados.
3. Devuelve una respuesta clara.
4. Incluye los enlaces de las fuentes utilizadas.
"""

agente_web = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

print("\n🧰 Herramientas registradas:")
for t in tools:
    print(f"  ✅ {t.name:<15} — {t.description}")
print("\n✅ Agente creado con create_react_agent.")


🧰 Herramientas registradas:
  ✅ busca_web       — Realiza una búsqueda en la web sobre un tema específico

✅ Agente creado con create_react_agent.


## Paso 5: Prueba de Herramientas

En este paso probaremos que las herramientas vinculadas al modelo funcionan correctamente de forma independiente.

Esto incluye:

- **busca_web**: el agente detecta que debe buscar en la web y retorna una respuesta elaborada con fuentes.

> **Nota:** el agente maneja el ciclo razonamiento → herramienta → respuesta de forma autónoma usando el patrón ReAct.


In [10]:
import re, html

def limpiar_texto(texto: str) -> str:
    texto = html.unescape(texto)
    texto = re.sub(r'\[.*?\]\(.*?\)', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# Prueba con búsqueda web
resultado = agente_web.invoke({
    "messages": [("user", "¿Cuáles son los impactos de la inteligencia artificial en la educación?")]
})

respuesta_final = resultado["messages"][-1]
c = respuesta_final.content
if isinstance(c, list):
    contenido = ' '.join(p['text'] for p in c if isinstance(p, dict) and 'text' in p)
else:
    contenido = c
print("\n🤖 Respuesta del agente:")
print("─" * 60)
print(limpiar_texto(contenido))


🤖 Respuesta del agente:
────────────────────────────────────────────────────────────
La inteligencia artificial (IA) está generando una transformación significativa en el ámbito educativo, ofreciendo tanto beneficios como desafíos importantes. **Impactos Positivos:** * **Personalización del aprendizaje:** La IA permite adaptar la enseñanza a las necesidades individuales de cada estudiante, mejorando los procesos educativos y ofreciendo enfoques renovados. Esto potencia la forma en que los estudiantes adquieren conocimientos y los maestros imparten sus clases. (LinkTIC, Universidad ORT Uruguay) * **Mejora de los procesos y recursos educativos:** La IA contribuye a la creación rápida de recursos educativos y al desarrollo de nuevas estrategias de enseñanza. Además, facilita la evolución constante y la mejora continua de los procesos pedagógicos. (LinkTIC, Galileo.edu) * **Abordaje de desafíos educativos:** La IA tiene el potencial de innovar las prácticas de enseñanza y aprendizaje, y a

## Paso 6: Agente Científico con ArXiv

En este paso crearemos un agente especializado en búsqueda de artículos académicos usando **ArXiv**.

- **tool_cientifica**: herramienta que consulta arXiv y retorna títulos, autores y resúmenes.
- **agente_cientifico**: agente ReAct con `create_react_agent` vinculado a la herramienta arXiv.

> **Nota:** ArXiv es un repositorio de acceso abierto con más de 2 millones de artículos científicos en física, matemáticas, computación y más.


In [14]:
import arxiv

@tool
def tool_cientifica(query: str) -> str:
    """Busca artículos científicos en arXiv sobre un tema específico"""
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=10)
    resultados = []
    for r in client.results(search):
        resultados.append(f"- {r.title} ({r.published.year})\n  {r.entry_id}")
    return "\n".join(resultados) if resultados else "No se encontraron resultados."

system_prompt2 = """
Actúa como un asistente útil.
Usa las herramientas proporcionadas para responder a las preguntas del usuario.

- tool_cientifica: Retorna resultados de una búsqueda en arXiv.

Cuando el usuario pregunte sobre un tema específico, usa tool_cientifica y devuelve los títulos de los artículos encontrados.
"""

agente_cientifico = create_react_agent(
    model=llm,
    tools=[tool_cientifica],
    prompt=system_prompt2
)

resultado = agente_cientifico.invoke({
    "messages": [
        ("user", "AI impact in education")
    ]
})

contenido = resultado["messages"][-1].content
if isinstance(contenido, list):
    print(contenido[0]["text"])
else:
    print(contenido)

Aquí hay algunos artículos sobre el impacto de la IA en la educación:

- Need of AI in Modern Education: in the Eyes of Explainable AI (xAI) (2024)
- Exploring utilization of generative AI for research and education in data-driven materials science (2025)
- AI & Data Competencies: Scaffolding holistic AI literacy in Higher Education (2025)
- Developing Strategies to Increase Capacity in AI Education (2025)
- Use Scenarios & Practical Examples of AI Use in Education (2023)
- Bridging Technical AI, Societal Impacts, and Workforce Competencies in AI Education (2026)
